<a id="s1"></a>

#### LLM Fundamentals Playground

এটা Phase 1 (W01-W04)-এর project। এখানে Prompt-এর token গোনা, পরের token-এর সম্ভাবনা, temperature ও sampling নিজের হাতে পরীক্ষা করা হয়। এখানে model-এর score হাতে বানানো, আসল model-এর উত্তর W05-এ API এলে যোগ হবে।

## সূচিপত্র

- [LLM Fundamentals Playground](#s1)
- [Token গোনা: ইংরেজি আর বাংলা](#s2)
- [Token-এর ভেতরে: byte আর টুকরো](#s3)
- [Token Counter Function](#s4)
- [Softmax: Score থেকে সম্ভাবনা](#s5)
- [Temperature](#s6)
- [Sampling: সম্ভাবনা ধরে বাছাই](#s7)
- [সব একসাথে: playground function](#s8)
- [Skill Gate: নিজের ভাষায় ব্যাখ্যা](#s9)
- [LLM কীভাবে পরের token অনুমান করে](#s10)

<a id="s2"></a>

#### Token গোনা: ইংরেজি আর বাংলা

#### আগে একটা গল্প
মোবাইলে একটা SMS-এ ইংরেজিতে প্রায় ১৬০ অক্ষর লেখা যায়, কিন্তু বাংলায় লেখা যায় প্রায় ৭০ অক্ষর। কথা একই, অথচ বাংলায় বেশি SMS লাগে। LLM-এর বেলায়ও এমন একটা হিসাব চলে, শুধু একক অক্ষর নয়, **token**।

#### মূল ধারণা
LLM লেখা সরাসরি পড়ে না। **Tokenizer** আগে লেখাকে ছোট ছোট টুকরোয় ভাঙে, প্রতিটা টুকরোর নাম **token**, আর প্রতিটা token একটা নম্বর পায়। এই Playground-এ OpenAI-র `cl100k_base` tokenizer ব্যবহার হয়েছে, `tiktoken` library দিয়ে।

একই অর্থের দুটো বাক্য দেখো। `I eat rice` ভাঙলে **৩টা** token হয়। `আমি ভাত খাই` ভাঙলে **১৩টা** token হয়। বাংলায় প্রায় **চার গুণ** (১৩ ÷ ৩ ≈ ৪.৩)।

এতে দুটো জিনিস বদলায়। প্রথমত, **খরচ**, কারণ API-র হিসাব সাধারণত token ধরে হয়। দ্বিতীয়ত, **context window-র জায়গা**, কারণ context window-র সীমাও token ধরে মাপা হয়। একই কথা বাংলায় লিখলে খরচ বেশি লাগে আর জায়গা কম থাকে।

**Demo:** প্রথম cell-এ `tiktoken` install হয়। দ্বিতীয় cell-এ দুটো বাক্যের token গোনা হয়েছে।

#### মনে রাখার এক লাইন
> খরচ আর context window দুটোই token ধরে মাপা হয়, আর বাংলায় একই কথায় ইংরেজির চেয়ে অনেক বেশি token লাগে।

In [67]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [68]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

for text in ["I eat rice", "আমি ভাত খাই"]:
    print(f"{len(enc.encode(text)):2d} token: {text}")

 3 token: I eat rice
13 token: আমি ভাত খাই


<a id="s3"></a>

#### Token-এর ভেতরে: byte আর টুকরো

#### আগে একটা গল্প
মুদি দোকানে যা সবচেয়ে বেশি বিক্রি হয়, তার জন্য আগে থেকেই প্যাকেট বানানো থাকে, যেমন এক কেজি চিনি। কিন্তু যে জিনিস কেউ চায় না, তার প্যাকেট নেই। তখন দোকানদার সেটা খুচরো দানায় মেপে দেয়। Tokenizer-এও ইংরেজি শব্দের তৈরি প্যাকেট আছে, বাংলার অনেক অক্ষরের নেই।

#### মূল ধারণা
কম্পিউটার প্রতিটা অক্ষর **byte** দিয়ে রাখে। ইংরেজি `a` মাত্র **১ byte** (`b'a'`)। বাংলা `আ` লাগে **৩ byte** (`\xe0\xa6\x86`)।

Tokenizer-এর **vocabulary** মূলত ইংরেজি লেখা থেকে বানানো। তাই চালু ইংরেজি শব্দ একটা পুরো token হয়ে যায়। বাংলার অক্ষর ভেঙে byte-এর টুকরো হয়, আর একটা অক্ষরের তিন byte সবসময় একই token-এ থাকে না।

এর ফল হলো, একা একটা token decode করলে অনেক সময় অর্ধেক byte পাওয়া যায়, যার কোনো অর্থ হয় না। তখন `�` আসে। কিন্তু সব token একসাথে `enc.decode(tokens)` করলে byte-গুলো আবার জোড়া লাগে, আর পুরো বাক্য `আমি ভাত খাই` ফিরে আসে।

**Demo:** নিচের cell-এ প্রতিটা token আলাদা করে decode করে দেখা হয়েছে। এরপর `a` আর `আ`-র byte দেখানো হয়েছে। শেষ লাইনে সব token একসাথে decode করা হয়েছে, আর সেটাই প্রমাণ যে কিছু হারায়নি।

#### মনে রাখার এক লাইন
> বাংলার অক্ষর ভেঙে byte-এর টুকরো হয়, তাই একা একটা token অনেক সময় পড়া যায় না (`�`); সব token একসাথে decode করলে পুরো বাক্য ফেরে।

In [69]:
import tiktoken
import numpy as np

enc = tiktoken.get_encoding("cl100k_base")


# text = "I eat rice"
# tokens= enc.encode(text)
text_2 = "আমি ভাত খাই"
tokens= enc.encode(text_2)

for t in tokens:
    print(t, repr(enc.decode([t])))

print(len(tokens), tokens)
print("a".encode("utf-8"))
print("আ".encode("utf-8"))
print(enc.decode(tokens))



11372 '�'
228 '�'
11372 '�'
106 '�'
62456 'ি'
36278 ' �'
255 '�'
50228 'া�'
97 '�'
36278 ' �'
244 '�'
50228 'া�'
229 '�'
13 [11372, 228, 11372, 106, 62456, 36278, 255, 50228, 97, 36278, 244, 50228, 229]
b'a'
b'\xe0\xa6\x86'
আমি ভাত খাই


<a id="s4"></a>

#### Token Counter Function

#### আগে একটা গল্প
দোকানে প্রতিবার হাতে হিসাব না কষে একটা মাপার যন্ত্র রাখা থাকে। যেকোনো জিনিস যন্ত্রে ফেললেই ওজন বেরিয়ে আসে। লেখার token গোনার জন্যও এমন একটা যন্ত্র বানিয়ে রাখা যায়।

#### মূল ধারণা
`count_token(text)` function একটা লেখা নিয়ে তিনটা মান ফেরত দেয়: **শব্দ কয়টা**, **token কয়টা**, আর **গড়ে প্রতি শব্দে কয় token**। গড় যত বেশি, লেখাটা তত বেশি জায়গা আর খরচ নেয়।

নিচের উদাহরণে বাংলা বাক্যটায় ৪টা শব্দের জন্য ৩৬টা token লাগে, গড় ৯.০০। ফল ধরার জন্য নাম রাখা হয়েছে `word_count`, `token_count` আর `average_token`। `words` নামটা নেওয়া হয়নি, কারণ পরের ধাপে `words` নামে শব্দের তালিকা আসবে, আর একই নাম দুই কাজে ব্যবহার করলে একটা অন্যটাকে মুছে দেয়।

**Demo:** নিচের cell-এ function বানিয়ে একটা বাংলা বাক্যে চালানো হয়েছে, আর ফল সাজিয়ে print করা হয়েছে।

#### মনে রাখার এক লাইন
> শব্দ নয়, token-ই খরচ আর জায়গার একক; গড় token প্রতি শব্দ দেখলেই বোঝা যায় লেখাটা কতটা ভারী।

In [70]:
enc = tiktoken.get_encoding("cl100k_base")

def count_token(text):
    words = len(text.split())
    tokens = len(enc.encode(text))
    average_token = tokens/words
    return words, tokens, average_token

word_count, token_count, average_token = count_token("আউটপুটে চারটা সংখ্যা আসবে")

print(f"শব্দ: {word_count}")
print(f"token: {token_count}")
print(f"গড়: {average_token:.2f} token প্রতি শব্দ")

শব্দ: 4
token: 36
গড়: 9.00 token প্রতি শব্দ


<a id="s5"></a>

#### Softmax: Score থেকে সম্ভাবনা

#### আগে একটা গল্প
চারজন প্রার্থী নম্বর পেল ৪, ৩, ২ আর ১। নম্বর দেখে বলা যায় কে এগিয়ে, কিন্তু কার জেতার সম্ভাবনা কত শতাংশ, তা বলা যায় না। শতাংশ পেতে হলে নম্বরগুলোকে এমন সংখ্যায় বদলাতে হবে, যাদের যোগফল ১ (মানে ১০০%)।

#### মূল ধারণা
"আমি ভাত ..." লেখার পর পরের শব্দ কী হবে? Model প্রতিটা সম্ভাব্য শব্দকে একটা **score** দেয়। এখানে চারটা শব্দের score হাতে বানানো: `খাই` ৪, `খাচ্ছি` ৩, `রাঁধি` ২, `ঘুমাই` ১।

**Softmax** score-কে সম্ভাবনায় বদলায়। প্রতিটা score-এর `exp` নিয়ে সবকটার যোগফল দিয়ে ভাগ করা হয়। তাই প্রতিটা সম্ভাবনা ০ আর ১-এর মাঝে থাকে, বড় score-এ বড় সম্ভাবনা, আর সব সম্ভাবনার **যোগফল ১**।

Function-এ `scores / temperature` লাইনটা পরের ধাপের জন্য। আর `scaled.max()` বিয়োগ করা হয় যাতে বড় সংখ্যায় `exp` উপচে না যায়, এতে ফল বদলায় না। Temperature ১.০ হলে সম্ভাবনা দাঁড়ায় প্রায় ০.৬৪৪, ০.২৩৭, ০.০৮৭ আর ০.০৩২।

**Demo:** প্রথম cell-এ function, দ্বিতীয় cell-এ শব্দ, score আর তার সম্ভাবনা।

#### মনে রাখার এক লাইন
> Softmax score-কে সম্ভাবনায় বদলায়: বড় score-এ বড় সম্ভাবনা, আর সব সম্ভাবনার যোগফল ১।

In [71]:
import numpy as np

def softmax_with_temperature(scores, temperature):
    scaled = scores / temperature
    exp_values = np.exp(scaled - scaled.max())
    return exp_values / exp_values.sum()



In [72]:
words = ["খাই", "খাচ্ছি", "রাঁধি", "ঘুমাই"]
scores = np.array([4.0, 3.0, 2.0, 1.0])

probs = softmax_with_temperature(scores, 1.0)

print(probs.round(3))

[0.644 0.237 0.087 0.032]


<a id="s6"></a>

#### Temperature

#### আগে একটা গল্প
পরীক্ষায় একজন ছাত্র সবচেয়ে পরিচিত উত্তরটাই লেখে, সে সাবধানী। আরেকজন নতুন কিছু চেষ্টা করে, সে ঝুঁকি নেয়। একই ছাত্রকে কখনো সাবধানী, কখনো সাহসী হতে বলার একটা নব থাকলে সেটাই temperature।

#### মূল ধারণা
**Temperature** হলো softmax-এর আগে score-কে যে সংখ্যা দিয়ে ভাগ করা হয়। Temperature আমরা নিজে ঠিক করে দিই, model নিজে ঠিক করে না।

**কম temperature (০.৫):** বড় score আরও বড় হয়ে যায়, তাই সবচেয়ে সম্ভাব্য শব্দ প্রায় সবসময় জেতে। এখানে `খাই`-এর সম্ভাবনা ০.৮৬৫, মানে নিরাপদ আর একঘেয়ে।

**মাঝারি temperature (১.০):** score যেমন আছে তেমন। `খাই` ০.৬৪৪।

**বেশি temperature (২.০):** ব্যবধান কমে যায়, সম্ভাবনাগুলো কাছাকাছি আসে। `খাই` নেমে আসে ০.৪৫৫-এ, আর বাকিগুলো ০.২৭৬, ০.১৬৭ আর ০.১০২, মানে এলোমেলো।

**Demo:** নিচের cell-এ একই score তিনটা temperature-এ চালানো হয়েছে।

#### মনে রাখার এক লাইন
> Temperature কম হলে উত্তর নিরাপদ ও একই ধরনের, বেশি হলে সম্ভাবনা কাছাকাছি এসে উত্তর এলোমেলো হয়।

In [73]:
for temperature in [0.5, 1.0, 2.0]:
    probs = softmax_with_temperature(scores, temperature)
    print(temperature, probs.round(3))

0.5 [0.865 0.117 0.016 0.002]
1.0 [0.644 0.237 0.087 0.032]
2.0 [0.455 0.276 0.167 0.102]


<a id="s7"></a>

#### Sampling: সম্ভাবনা ধরে বাছাই

#### আগে একটা গল্প
একটা বাক্সে ১০০টা টিকিট আছে। ৬৪টায় "খাই", ২৪টায় "খাচ্ছি", ৯টায় "রাঁধি", আর ৩টায় "ঘুমাই" লেখা। চোখ বন্ধ করে একটা টিকিট তুললে বেশিরভাগ সময় "খাই" ওঠে, কিন্তু সবসময় ওঠে না।

#### মূল ধারণা
সম্ভাবনা পাওয়ার পর model-কে একটা শব্দ বেছে নিতে হয়। এই বাছাইকে বলে **sampling**। `np.random.choice(words, p=probs)` ঠিক সেই লটারির কাজ করে: সম্ভাবনা ধরে একটা শব্দ বাছে।

সবচেয়ে সম্ভাব্য শব্দটাই বাছা হয় না, বেশি সম্ভাবনার শব্দ শুধু বেশিবার ওঠে। তাই একই প্রশ্নে চালালে উত্তর আলাদা হতে পারে।

১০০০ বার চালালে গোনা সংখ্যাগুলো সম্ভাবনার কাছাকাছি আসে, প্রায় ৬৪৪, ২৩৭, ৮৭ আর ৩২। প্রতিবার সংখ্যা সামান্য বদলাবে, কারণ বাছাই এলোমেলো। `np.str_('খাই')` মানে NumPy-র string, শব্দ একই।

**Demo:** প্রথম cell-এ একবার বাছা হয়েছে। দ্বিতীয় cell-এ `Counter` দিয়ে ১০০০ বারের গণনা।

#### মনে রাখার এক লাইন
> Sampling মানে সম্ভাবনা ধরে লটারি, তাই একই প্রশ্নে উত্তর আলাদা হতে পারে; বেশি বার চালালে গণনা সম্ভাবনার কাছাকাছি আসে।

In [74]:
probs = softmax_with_temperature(scores, 1.0)
chosen = np.random.choice(words, p=probs)
print(chosen)

রাঁধি


In [75]:
from collections import Counter

samples = [np.random.choice(words, p=probs) for _ in range(1000)]
print(Counter(samples))

Counter({np.str_('খাই'): 643, np.str_('খাচ্ছি'): 226, np.str_('রাঁধি'): 101, np.str_('ঘুমাই'): 30})


<a id="s8"></a>

#### সব একসাথে: playground function

#### আগে একটা গল্প
কারখানায় আলাদা আলাদা যন্ত্র একটা লাইনে জোড়া থাকলে কাঁচামাল ঢুকিয়ে একবারে তৈরি জিনিস পাওয়া যায়। এতক্ষণ যে চারটা ধাপ আলাদা করে দেখা হলো, তাদের একটা লাইনে জুড়লেই `playground` হয়।

#### মূল ধারণা
`playground(text, temperature)` তিনটা কাজ করে। `text`-এর token গোনে, temperature দিয়ে softmax-এর সম্ভাবনা বানায়, আর সম্ভাবনা ধরে একটা শব্দ বাছে। ফেরত দেয় `token_count`, `probs` আর `chosen`।

একটা সীমা মনে রাখতে হবে। `scores` হাতে বানানো আর স্থির, তাই `text` বদলালে শুধু token সংখ্যা বদলায়, সম্ভাবনা বদলায় না। আসল model লেখা পড়ে score বানায়, সেটা W05-এ API এলে দেখা যাবে।

**Demo:** প্রথম cell-এ `"আমি ভাত"` একবার চালানো হয়েছে temperature ১.০-এ। দ্বিতীয় cell-এ একই লেখা তিন temperature-এ চালানো হয়েছে। কম temperature-এ `খাই` প্রায় সবসময় আসে, বেশি temperature-এ অন্য শব্দও আসার সুযোগ পায়। কয়েকবার চালিয়ে দেখো।

#### মনে রাখার এক লাইন
> Playground-এ token গোনা, softmax, temperature আর sampling একসাথে জোড়া; এর score হাতে বানানো, আসল model-এর নয়।

In [76]:
def playground(text, temperature):
    token_count = len(enc.encode(text))
    probs = softmax_with_temperature(scores, temperature)
    chosen = np.random.choice(words, p=probs)
    return token_count, probs, chosen

token_count, probs, chosen = playground("আমি ভাত", 1.0)
print("token:", token_count)
print("সম্ভাবনা:", probs.round(3))
print("বাছা শব্দ:", chosen)

print(words)

token: 9
সম্ভাবনা: [0.644 0.237 0.087 0.032]
বাছা শব্দ: খাই
['খাই', 'খাচ্ছি', 'রাঁধি', 'ঘুমাই']


In [77]:
for temperature in [0.5, 1.0, 2.0]:
    token_count, probs, chosen = playground("আমি ভাত", temperature)
    print(f"temperature {temperature}: token {token_count}, সম্ভাবনা {probs.round(3)}, বাছা শব্দ {chosen}")

temperature 0.5: token 9, সম্ভাবনা [0.865 0.117 0.016 0.002], বাছা শব্দ খাই
temperature 1.0: token 9, সম্ভাবনা [0.644 0.237 0.087 0.032], বাছা শব্দ খাচ্ছি
temperature 2.0: token 9, সম্ভাবনা [0.455 0.276 0.167 0.102], বাছা শব্দ খাই


<a id="s9"></a>

#### Skill Gate: নিজের ভাষায় ব্যাখ্যা

<a id="s10"></a>

#### LLM কীভাবে পরের token অনুমান করে

১. লেখা model-এর কাছে যাওয়ার আগে tokenizer সেটাকে ছোট ছোট টুকরো (token) করে নম্বরে বদলায়। বাংলায় একই কথায় ইংরেজির চেয়ে অনেক বেশি token লাগে।

২. প্রতি ধাপে model পরের সম্ভাব্য প্রতিটা token-এর একটা score বের করে। Softmax সেই score-কে সম্ভাবনায় বদলায়, যাদের যোগফল ১।

৩. Temperature আমরা ঠিক করে দিই। কম হলে সবচেয়ে সম্ভাব্য token প্রায় সবসময় জেতে, বেশি হলে সম্ভাবনাগুলো কাছাকাছি আসে।

৪. তারপর সম্ভাবনা ধরে লটারির মতো একটা token বাছা হয়। তাই একই প্রশ্নে অর্থ এক থাকলেও উত্তর আলাদা আসতে পারে।

৫. Model সত্য যাচাই করে না, শুধু সম্ভাব্য token অনুমান করে। তাই কখনো কখনো আত্মবিশ্বাসের সাথে ভুল (hallucination) বলে। গুরুত্বপূর্ণ তথ্য, বিশেষ করে সংখ্যা, নিজে যাচাই করতে হবে।